<a href="https://colab.research.google.com/github/nurhikmahsalam3-creator/Tugas-4-Sistem-temu-kembali/blob/main/240210500015_NUR_HIKMAH_SALAM_PREPROCESSING.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**TUGAS 4**

NAMA: NUR HIKMAH SALAM

NIM: 240210500015

MATA KULIAH: SISTEM TEMU KEMBALI

TOPIK: TOKENISASI, STOPWORDS REMOVAL, DAN STEMMING


**NOMOR 1 - PREPOCESSING TEKS**

In [12]:
### Instalasi & Import Library

# Install library Sastrawi (menyediakan stemmer & stopword remover Bahasa Indonesia)
# Tanda "!" di depan berarti perintah dijalankan di terminal Colab, bukan kode Python biasa
!pip install Sastrawi -q

In [13]:
import re  # library bawaan Python untuk regular expression (mencocokkan/menghapus pola teks)
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory              # untuk mengubah kata berimbuhan -> kata dasar
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory  # untuk daftar kata umum (stopwords)

# Buat objek stemmer sekali saja di sini, supaya bisa dipakai berulang kali tanpa membuat ulang
stemmer_factory = StemmerFactory()
stemmer = stemmer_factory.create_stemmer()

# Ambil daftar stopwords bawaan Sastrawi, ubah ke set() agar pengecekan keanggotaan lebih cepat
stopword_factory = StopWordRemoverFactory()
stopwords_set = set(stopword_factory.get_stop_words())

In [14]:
### DATASET

# Kumpulan 5 dokumen mentah (masih "kotor": ada huruf kapital, tanda baca, angka, stopwords)
dokumen = [
    # Dokumen 1
    """Fakultas Teknik UNM mengumumkan bahwa pendaftaran praktikum Pengolahan Citra
    Digital untuk semester ini telah dibuka! Mahasiswa diharapkan segera mendaftarkan
    diri melalui portal akademik sebelum tanggal 15 September 2026.""",

    # Dokumen 2
    """Perkembangan teknologi kecerdasan buatan (Artificial Intelligence) semakin pesat
    di tahun 2026. Banyak perusahaan teknologi besar berlomba-lomba mengembangkan model
    bahasa raksasa (Large Language Model) untuk berbagai keperluan bisnis dan penelitian.""",

    # Dokumen 3
    """Laboratorium Komputer Fakultas Teknik akan melakukan pemeliharaan rutin pada
    hari Sabtu, 20 September 2026. Selama pemeliharaan berlangsung, seluruh layanan
    laboratorium komputer tidak dapat diakses oleh mahasiswa maupun dosen.""",

    # Dokumen 4
    """Sistem temu kembali informasi merupakan cabang ilmu komputer yang mempelajari
    bagaimana cara menemukan dokumen relevan dari kumpulan data yang sangat besar.
    Teknik seperti TF-IDF dan pembobotan term menjadi dasar penting dalam bidang ini.""",

    # Dokumen 5
    """Pengumuman resmi dari bagian akademik: yudisium periode Oktober 2026 akan
    dilaksanakan secara daring melalui aplikasi Zoom. Mahasiswa yang telah menyelesaikan
    seluruh mata kuliah diwajibkan mengisi formulir pendaftaran yudisium paling lambat
    tanggal 5 Oktober 2026."""
]

# Tampilkan dulu semua dokumen mentah agar terlihat jelas kondisi "sebelum dibersihkan"
for i, doc in enumerate(dokumen, 1):
    print(f"Dokumen {i}:")
    print(doc.strip())
    print("-" * 80)


Dokumen 1:
Fakultas Teknik UNM mengumumkan bahwa pendaftaran praktikum Pengolahan Citra
    Digital untuk semester ini telah dibuka! Mahasiswa diharapkan segera mendaftarkan
    diri melalui portal akademik sebelum tanggal 15 September 2026.
--------------------------------------------------------------------------------
Dokumen 2:
Perkembangan teknologi kecerdasan buatan (Artificial Intelligence) semakin pesat
    di tahun 2026. Banyak perusahaan teknologi besar berlomba-lomba mengembangkan model
    bahasa raksasa (Large Language Model) untuk berbagai keperluan bisnis dan penelitian.
--------------------------------------------------------------------------------
Dokumen 3:
Laboratorium Komputer Fakultas Teknik akan melakukan pemeliharaan rutin pada
    hari Sabtu, 20 September 2026. Selama pemeliharaan berlangsung, seluruh layanan
    laboratorium komputer tidak dapat diakses oleh mahasiswa maupun dosen.
-------------------------------------------------------------------------------

In [15]:
### Fungsi preprocess_text

def preprocess_text(text):
    """
    Melakukan pipeline preprocessing teks Bahasa Indonesia secara berurutan:
    1. Case folding      -> menyamakan semua huruf jadi huruf kecil
    2. Cleaning           -> menghapus angka, tanda baca, dan karakter khusus
    3. Tokenisasi         -> memecah kalimat menjadi kata-kata (token)
    4. Stopwords removal  -> membuang kata umum yang tidak bermakna
    5. Stemming           -> mengubah kata berimbuhan menjadi kata dasar

    Parameter:
        text (str): teks/dokumen mentah yang akan diproses

    Return:
        tokens_before (list): token hasil tokenisasi SEBELUM stopwords removal & stemming
        tokens_after  (list): token akhir SESUDAH stopwords removal & stemming
    """

    # 1. Case folding: ubah semua huruf jadi huruf kecil,
    #    supaya "Sistem" dan "sistem" dianggap kata yang sama
    text = text.lower()

    # 2. Cleaning: hapus semua karakter SELAIN huruf a-z dan spasi
    #    (menghapus angka seperti "2026", tanda baca seperti "!", "()", dsb.)
    text = re.sub(r"[^a-z\s]", " ", text)

    # Rapikan spasi ganda/berlebih (akibat karakter yang dihapus) menjadi satu spasi saja
    text = re.sub(r"\s+", " ", text).strip()

    # 3. Tokenisasi: pecah teks yang sudah bersih menjadi daftar kata (token) berdasarkan spasi
    tokens_before = text.split()

    # 4. Stopwords removal: buang token yang termasuk kata umum/stopwords
    #    contoh yang dibuang: "yang", "dan", "akan", "untuk", "telah", dll.
    tokens_no_stop = [t for t in tokens_before if t not in stopwords_set]

    # 5. Stemming: ubah tiap token yang tersisa menjadi kata dasarnya
    #    contoh: "mendaftarkan" -> "daftar", "diakses" -> "akses"
    tokens_after = [stemmer.stem(t) for t in tokens_no_stop]

    # Kembalikan dua versi: token sebelum diproses penuh, dan token hasil akhir
    return tokens_before, tokens_after


In [16]:
### Penerapan fungsi ke-5 dokumen

hasil_preprocessing = []  # list untuk menampung hasil preprocessing tiap dokumen

for i, doc in enumerate(dokumen, 1):        # enumerate mulai dari 1 supaya penomoran dokumen alami
    tokens_before, tokens_after = preprocess_text(doc)  # panggil fungsi yang sudah dibuat sebelumnya

    # simpan semua info penting dokumen ke-i ke dalam satu dictionary
    hasil_preprocessing.append({
        "no": i,
        "dokumen_asli": doc.strip(),
        "tokens_before": tokens_before,
        "tokens_after": tokens_after,
        "jumlah_token_sebelum": len(tokens_before),   # jumlah token SEBELUM preprocessing penuh
        "jumlah_token_sesudah": len(tokens_after)     # jumlah token SESUDAH preprocessing penuh
    })

print("Preprocessing selesai untuk", len(hasil_preprocessing), "dokumen.")

Preprocessing selesai untuk 5 dokumen.


In [17]:
### Perbandingan sebelum dan sesudah

# Ambil 2 dokumen pertama saja untuk ditampilkan secara lengkap (slicing [:2])
for item in hasil_preprocessing[:2]:
    print(f"===== DOKUMEN {item['no']} =====")
    print("\nTeks Asli:")
    print(item["dokumen_asli"])
    print("\nToken SEBELUM preprocessing (hasil tokenisasi saja):")
    print(item["tokens_before"])
    print("\nToken SESUDAH preprocessing (stopwords removal + stemming):")
    print(item["tokens_after"])
    print("\nJumlah token sebelum :", item["jumlah_token_sebelum"])
    print("Jumlah token sesudah :", item["jumlah_token_sesudah"])
    print("=" * 80, "\n")


===== DOKUMEN 1 =====

Teks Asli:
Fakultas Teknik UNM mengumumkan bahwa pendaftaran praktikum Pengolahan Citra
    Digital untuk semester ini telah dibuka! Mahasiswa diharapkan segera mendaftarkan
    diri melalui portal akademik sebelum tanggal 15 September 2026.

Token SEBELUM preprocessing (hasil tokenisasi saja):
['fakultas', 'teknik', 'unm', 'mengumumkan', 'bahwa', 'pendaftaran', 'praktikum', 'pengolahan', 'citra', 'digital', 'untuk', 'semester', 'ini', 'telah', 'dibuka', 'mahasiswa', 'diharapkan', 'segera', 'mendaftarkan', 'diri', 'melalui', 'portal', 'akademik', 'sebelum', 'tanggal', 'september']

Token SESUDAH preprocessing (stopwords removal + stemming):
['fakultas', 'teknik', 'unm', 'umum', 'daftar', 'praktikum', 'olah', 'citra', 'digital', 'semester', 'buka', 'mahasiswa', 'harap', 'segera', 'daftar', 'diri', 'lalu', 'portal', 'akademik', 'tanggal', 'september']

Jumlah token sebelum : 26
Jumlah token sesudah : 21

===== DOKUMEN 2 =====

Teks Asli:
Perkembangan teknologi kece

In [18]:
### Statistik jumlah token & persentase Pengurangan

# Cetak header tabel dengan rata kiri menggunakan format string f"{teks:<lebar}"
print(f"{'Dokumen':<10}{'Token Sebelum':<16}{'Token Sesudah':<16}{'Pengurangan (%)':<16}")
print("-" * 58)  # garis pemisah header dan isi tabel

total_before = 0   # akumulator total token sebelum, dijumlahkan dari semua dokumen
total_after = 0    # akumulator total token sesudah

for item in hasil_preprocessing:
    # rumus persentase pengurangan token = (sebelum - sesudah) / sebelum * 100
    reduksi = (item["jumlah_token_sebelum"] - item["jumlah_token_sesudah"]) / item["jumlah_token_sebelum"] * 100

    # cetak satu baris statistik untuk dokumen ini, rata kiri agar rapi
    print(f"Dok {item['no']:<7}{item['jumlah_token_sebelum']:<16}{item['jumlah_token_sesudah']:<16}{reduksi:<16.2f}")

    # tambahkan ke akumulator total
    total_before += item["jumlah_token_sebelum"]
    total_after += item["jumlah_token_sesudah"]

# hitung persentase pengurangan total dari seluruh dokumen
total_reduksi = (total_before - total_after) / total_before * 100
print("-" * 58)
print(f"{'TOTAL':<10}{total_before:<16}{total_after:<16}{total_reduksi:<16.2f}")


Dokumen   Token Sebelum   Token Sesudah   Pengurangan (%) 
----------------------------------------------------------
Dok 1      26              21              19.23           
Dok 2      29              26              10.34           
Dok 3      26              21              19.23           
Dok 4      34              26              23.53           
Dok 5      31              26              16.13           
----------------------------------------------------------
TOTAL     146             120             17.81           


**ANALISIS**

Preprocessing teks (case folding, cleaning, tokenisasi, stopwords removal, dan stemming) terbukti secara signifikan mengurangi jumlah token
pada kelima dokumen dengan menghilangkan kata-kata umum yang tidak informatif (seperti "yang", "akan", "dan") serta menyeragamkan variasi
bentuk kata berimbuhan menjadi kata dasarnya melalui stemming. Dampaknya terhadap sistem temu kembali informasi (IR) sangat positif: ukuran
vocabulary menjadi lebih kecil sehingga indeks lebih efisien, term yang tersisa lebih diskriminatif dan merepresentasikan topik dokumen secara
lebih akurat (mendukung perhitungan TF-IDF pada tugas No. 2), serta variasi kata berimbuhan (misalnya "mendaftarkan" dan "pendaftaran") dapat
disatukan menjadi satu term dasar sehingga pencocokan query dengan dokumen menjadi lebih efektif dan tidak kehilangan dokumen relevan hanya
karena perbedaan bentuk kata.